In [13]:
import os
from dotenv import load_dotenv
load_dotenv()

True

# 1. init_chat_model

Initialize a chat model from any supported provider using a unified interface.

<b> Two main use cases:</b>
- Fixed model – specify the model upfront and get a ready-to-use chat model.
- Configurable model – choose to specify parameters (including model name) at runtime via config. Makes it easy to switch between models/providers without changing your code

In [14]:
from langchain.chat_models import init_chat_model

# Fixed Model Name
model = init_chat_model(
    model = "groq:openai/gpt-oss-20b",
    temperature = 0.7,
)
response = model.invoke("How many moons does Jupiter have?")
response.content

"Jupiter currently has **79 confirmed natural satellites** (moons). The count can change as new moons are discovered or reclassified, so it's always good to check the latest updates from sources like NASA or the IAU."

In [15]:
# Configurable
configurable_model = init_chat_model(
    temperature = 0.7
)

configurable_response = configurable_model.invoke(
    "How many moons does Earth have?",
    config = {
        "model": "groq:openai/gpt-oss-20b"
    }
)
configurable_response.content

'Earth has just one natural moon—our familiar Moon.'

Note: Parameter `configurable_fields` will allow to configure all fields at the runtime

In [16]:
# Configurable
configurable_model = init_chat_model(configurable_fields='any')

configurable_response = configurable_model.invoke(
    "How many moons does Earth have?",
    config = {
        "model": "groq:openai/gpt-oss-20b", "temperature": 0.7
    }
)
configurable_response.content

'Earth has one natural satellite—a single moon. Its name is simply “the Moon.”'

In [17]:
configurable_response = configurable_model.invoke(
    "How many moons does Earth have?",
    config = {
        "model": "google_genai:gemini-3.6-flash"
    }
)
configurable_response.content

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


[{'type': 'text',
  'text': 'Earth has **1** permanent natural moon, officially named **the Moon** (or *Luna*).\n\nHowever, Earth occasionally gets temporary "mini-moons"—small asteroids that are briefly captured by Earth\'s gravity for a few months or years before drifting away—as well as "quasi-satellites" (asteroids that orbit the Sun in a pattern very similar to Earth\'s).',
  'extras': {'signature': 'EpMKCpAKARFNMg+Qf6sbJgE3F5jJ9umko3mnDzjiULxBzT++5pJZuEe2dOdwznm9c7ZgnqCWHfyFOkGsOt9AXEBgXnHU93NTCwbCRWYf44zvFVS58ZJzfAQLumHwHm7Gcra8m1ynGCgoGP+FT2IsNhPiBssagUgQ3xNMtAW8dHkKWocBbUEQ11BqqDNPmuhoWeY79VRfvoLc7/kGpQTeOkyADOGdw2jWd9YnuNm0hmU6s2qHk9b43SYgePM5yHgs5xaFquIwt5hhUxGNDL9FrxmPmKaLKtM4BJO7c9Ap00SqYHLhRQZPJq8vGEXrea10bqHSOO0bCBubj8OSRhFVW11xl/nCfYHRcxrtKpeXpRlZbqwPYo7k4QKXUgUEx2lypAVVj7EfIsu1dlCYmUirWKTFp1S8IzPEEFfCYLDIIAq1g+BwvMFroriJtSge8JlFmRQZ35QBoHx7jMW8hxUQEZGAcsXdflEyqOK7m+6ZokT+pqGYAK66cjTmmUIoEFZf8hXp3M9CZzMPweAb7xKo6aMYZrVFEyjCE3oJKpveYTNuoLtd8Quj+1ETKJdZibt2MEQAX15hM8ywu4H

# 2. Streaming

> **Key idea:** Streaming sends the model response piece by piece as it is generated instead of waiting for the complete response.

#### Why use streaming?

Streaming improves the user experience because the first part of the answer can appear immediately. It is useful for chat applications, long responses, and agent workflows where users should see progress as it happens.

#### Basic model streaming

Use `model.stream(...)` to receive response chunks:

```python
for chunk in model.stream("Explain quantum computing in simple terms"):
    print(chunk.content, end="", flush=True)
```

Each `chunk` is a partial `AIMessage`. The complete response is formed by joining the `chunk.content` values.

#### `invoke` versus `stream`

| Method | Behavior | Best for |
|---|---|---|
| `invoke()` | Returns one complete response | Simple requests and batch processing |
| `stream()` | Returns response chunks progressively | Interactive chat interfaces |

#### Important notes

- A streamed chunk is not always a complete sentence or word.
- Use `end=""` to avoid extra newlines between chunks.
- Use `flush=True` so each chunk appears immediately in a terminal or notebook output.
- Streaming changes how the answer is delivered, not the model's final answer.
- The selected provider and model must support streaming.


In [20]:
for chunk in model.stream("How to drive a car?"):
    print(chunk.content, end="", flush=True)

**Driving a Car – A Step‑by‑Step Guide**

*(This is a general overview. Always follow the specific rules, regulations, and licensing requirements in your country or region. If you’re new to driving, consider enrolling in a certified driver‑education course and practice with a licensed instructor.)*

---

## 1. Before You Get Behind the Wheel

| ✅ | Checklist |
|---|-----------|
| 📜 **License** | Have a valid driver’s license (or learner’s permit) for the vehicle type you’ll drive. |
| 🛠️ **Vehicle Condition** | Check that the car is in good working order: brakes, lights, tires, fluid levels, and the horn. |
| ⚖️ **Insurance & Registration** | Ensure the car is insured and registered; carry proof in the vehicle. |
| 👀 **Seat & Mirrors** | Adjust the seat so you can reach all controls comfortably. Adjust all mirrors for a clear field of view. |
| 🪑 **Seatbelt** | Fasten your seatbelt (and any passenger seatbelts). |

---

## 2. Familiarize Yourself With the Controls

| Control | Purpose 

# 3. Batch

Batching a collection of independent requests ti a model can significantly improve performance and reduce costs, as the processing can be done in parallel.

> **Key idea:** `batch()` sends multiple independent inputs to the model and returns their responses together.

```python
responses = model.batch([
    "How many moons does Earth have?",
    "How many moons does Mars have?",
])

for response in responses:
    print(response.content)
```

Use `invoke()` for one input, `batch()` for multiple inputs, and `stream()` when the response should be displayed progressively.

In [25]:
responses = model.batch(
    [
        'how to divide 412 by 2?',
        'how do aeroplane fly?',
        'how many planets are around the sun?'
    ]
)

for response in responses:
    print(response.content, end="\n"*3+"*"*400+"\n"*3)

To divide 412 by 2, simply split the number in half:

```
412 ÷ 2 = 206
```

**Quick check:**
- 200 × 2 = 400  
- 6 × 2 = 12  
- 400 + 12 = 412

So the quotient is **206**. If you prefer a step‑by‑step method (long division), let me know!


****************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************


## Short answer  
An airplane flies because its wings are shaped to make the air above them move faster than the air below, creating a **lift** force that pushes the airplane up.  
At the same time the airplane’s engine (propeller or jet) gives it a **thrust** force that pushes it forward, overcoming the **drag** that resists its motion.  
When lift bala